# Problem 2: Frontier Exploration
All of your code goes in `P2_frontier.py`. The occupancy grid class and the simulated environment are in `utils.py` (you do not need to change it).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt

from utils import ground_truth, sense, make_occupancy, RESOLUTION
from P2_frontier import window_counts, explore, closest_frontier

os.makedirs("plots", exist_ok=True)
occ_gt = ground_truth()
start = np.array([1.5, 2.0])


def show(ax, probs, title):
    # unknown gray, free white, occupied black
    shade = np.where(probs < 0, 0.6, np.where(probs >= 50, 0.0, 1.0))
    ax.imshow(shade, origin="lower", cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("kx")
    ax.set_ylabel("ky")

## Ground truth and the first scan
The robot starts at (1.5, 2) m and senses everything it can see within 3 m. Unknown cells are -1 (gray), free cells 0 (white), and occupied cells 100 (black).

In [ ]:
probs = sense(-np.ones(occ_gt.shape), occ_gt, start)
occupancy = make_occupancy(probs)

frontier = explore(occupancy)
goal, dist = closest_frontier(frontier, start)
print("number of frontier cells:", len(frontier))
print("closest frontier state:", goal, " distance:", round(dist, 3), "m")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
show(axes[0], np.where(occ_gt, 100, 0), "ground truth")
show(axes[1], probs, "occupancy grid after the first scan")
k = frontier / RESOLUTION
axes[1].plot(k[:, 0], k[:, 1], "b+", ms=4, label="frontier")
axes[1].plot(start[0] / RESOLUTION, start[1] / RESOLUTION, "r*", ms=12, label="robot")
axes[1].plot(goal[0] / RESOLUTION, goal[1] / RESOLUTION, "ro", mfc="none", ms=12, label="closest frontier")
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("plots/p2_frontier.png", dpi=120)
plt.show()

## Exploring the whole map
The robot repeatedly drives to the closest frontier state and scans again, until no frontier is left. Here it simply jumps to the goal; on the TurtleBot the navigator from HW2 plans the path there.

In [ ]:
state = start.copy()
probs = sense(-np.ones(occ_gt.shape), occ_gt, state)
path, traveled = [state.copy()], 0.0
num_free = (~occ_gt).sum()
for step in range(30):
    goal, dist = closest_frontier(explore(make_occupancy(probs)), state)
    if goal is None:
        break
    traveled += dist
    state = goal
    path.append(state.copy())
    probs = sense(probs, occ_gt, state)

known_free = ((probs >= 0) & (probs < 50)).sum() / num_free
print(f"steps: {len(path) - 1}, distance traveled: {traveled:.2f} m, free space mapped: {known_free:.3f}")

path = np.array(path) / RESOLUTION
fig, ax = plt.subplots(figsize=(7, 5.5))
show(ax, probs, "map after exploration")
ax.plot(path[:, 0], path[:, 1], "r.-")
for i, (x, y) in enumerate(path):
    ax.text(x + 1, y + 1, str(i), color="red", fontsize=8)
plt.tight_layout()
plt.savefig("plots/p2_exploration.png", dpi=120)
plt.show()